In [1]:
import spatialdata as sd
import spatialdata_plot
import sopa
import geopandas as gp
from pathlib import Path
import numpy as np

d:\Dom\Virtual_Environments\napari_registration_project\.venv\Lib\site-packages\dask\dataframe\__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
d:\Dom\Virtual_Environments\napari_registration_project\.venv\Lib\site-packages\xarray_schema\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
d:\Dom\Virtual_Environments\napari_registration_project\.venv\Lib\site-packages\spatialdata\_core\query\relational_query.py:530: FutureWarning: functools.partial will be a method descriptor in fu

In [3]:
#Read Xenium file
sdata = sopa.io.xenium(r"D:\Dom\Psoriasis project\4th year data\Second Round Data\Xenium outputs - second round\Resegmented Xenium Outputs\Ctrl",
                       cells_boundaries = True,
                       cells_table = True,
                       nucleus_labels = True,
                       cells_labels = True,
                       nucleus_boundaries = True
                       )
sdata

C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3568.0_x64__qbz5n2kfra8p0\Lib\functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


SpatialData object
├── Images
│     └── 'morphology_focus': DataTree[cyx] (4, 13706, 51100), (4, 6853, 25550), (4, 3426, 12775), (4, 1713, 6387), (4, 856, 3193)
├── Labels
│     ├── 'cell_labels': DataTree[yx] (13706, 51100), (6853, 25550), (3426, 12775), (1713, 6387), (856, 3193)
│     └── 'nucleus_labels': DataTree[yx] (13706, 51100), (6853, 25550), (3426, 12775), (1713, 6387), (856, 3193)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 13) (3D points)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (38467, 1) (2D shapes)
│     └── 'nucleus_boundaries': GeoDataFrame shape: (34914, 1) (2D shapes)
└── Tables
      └── 'table': AnnData (38467, 5056)
with coordinate systems:
    ▸ 'global', with elements:
        morphology_focus (Images), cell_labels (Labels), nucleus_labels (Labels), transcripts (Points), cell_boundaries (Shapes), nucleus_boundaries (Shapes)

## Labelling cells according to their compartment

In [9]:
#Create a dictionary with compartment names and paths = {'compartment_name':'path/to/geojson'}
compartment_paths = {}
compartment_seg_folder = Path(r"D:\Dom\Psoriasis project\4th year data\Second Round Data\Xenium outputs - second round\output-XETG00160__0093234__ctrl__20260211__140710\Compartment segmentations")

for file in compartment_seg_folder.glob("*.geojson"): #get all geojson files
  name = file.stem.replace(" ", "_") 
  compartment_paths[name] = str(file)

#Read geojson files and create a dictionary containing compartment names and geodataframe shape objects = {'compartment_name': shape object}
compartment_shapes = {}
for compartment, path in compartment_paths.items():
  geojson = gp.read_file(path)
  shape = geojson.geometry.union_all #use in case of multiple polygons contained in the same file.
  compartment_shapes[compartment] = shape

#add shapes to sdata object for future reference
for compartment, shape in compartment_shapes.items():
  sdata.shapes[r'{compartment}'] = shape

#create a new column to identify which compartment a cell belongs to
adata = sdata.tables['table']

cell_compartment_locations = {}
for cell in adata['cellID']:
  #get cell x,y coordinates
  (x,y) = adata.loc[adata['cellID'] == cell, ['x', 'y']]

  #check which shape the x,y values land in
  for compartment, shape in compartment_shapes.items():
    if (x,y) in shape.bounds:
      cell_compartment_locations[cell] = compartment
    else:
      cell_compartment_locations[cell] = 'out_of_bounds'

adata['compartment'].map = cell_compartment_locations



sdata.tables['table'] = adata #reassign table to sdata

ValueError: Name must contain only alphanumeric characters, underscores, dots and hyphens.

## Separating Cartilage vs non-cartilage cell

In [ ]:
#For cells of a given Compartment selection, calculate median transcripts
cartilage_index = adata['Compartment' == 'Cartilage']
non_cartilage_index = adata['Compartment' != 'Cartilage' | 'Compartment' != 'out_of_bounds']

cartilage_df = adata[cartilage_index]
non_cartilage_df = adata[non_cartilage_index]

cartilage_median = np.median(cartilage_df['transcripts_count'])
non_cartilage_median = np.median(non_cartilage_df['transcripts_count'])

print(f'Median transcripts/cell for cartilage sections: {cartilage_median}')
print(f'Median transcripts/cell for non-cartilage sections: {non_cartilage_median}')